In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VG360-PALMID v1.0 — monochrome laptop-camera face + palm detection and palmprint identification
#   1. Capture colour webcam frames → colour TEACHERS label them (YuNet face, MediaPipe palm)
#   2. Train a MONOCHROME student (YOLOX-Nano, 2 classes) on the grayscale copies with SAM
#      (measured: MediaPipe palm detector drops from 54 → 5 detections on grayscale; the student fixes that)
#   3. Palmprint ID (CompCode, 6-orientation Gabor) on mono palm ROIs — enrolled, consenting people only;
#      open-set: anyone not enrolled is reported as "unknown". Templates = bit codes, never images.
#   Stack: Lean 4.31 kernel proofs · SAM · MCC + Mondrian SCCP · ONNX INT8 · OpenAI /v1/chat/completions
#   Note: a webcam resolves palm principal lines/creases, NOT friction ridges (those need ~500 dpi scanners).
# ═══════════════════════════════════════════════════════════════════════════════
import os, sys, json, time, math, glob, base64, shutil, asyncio, threading, subprocess, importlib, re, urllib.request

IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('VG360_SMOKE', '0') == '1'
WORK = '/content/vg360_palmid' if IN_COLAB else os.path.abspath('vg360_palmid_work')
os.makedirs(WORK, exist_ok=True)

CFG = dict(
    source=os.environ.get('VG360_SOURCE', 'webcam'),   # 'webcam' | video/gif path | glob of images
    classes=['face', 'palm'], input_size=416,
    collect_frames=int(os.environ.get('VG360_COLLECT', 60 if SMOKE else 300)), collect_every=1,   # teacher-labelled frames to gather
    collect_seconds=150, cam_w=640, cam_h=480, grab_batch=8, grab_ms=60, preview_every=4, progress_every=20,
    epochs=int(os.environ.get('VG360_EPOCHS', 2 if SMOKE else 30)), batch=8 if SMOKE else 16, lr=0.01, momentum=0.9, wd=5e-4,
    warmup_iters=10 if SMOKE else 200, sam_rho=0.05, sam_adaptive=False,
    teacher_face_thr=0.7, teacher_palm_thr=0.6,
    n_cal_bound=29, tau_fallback=0.30, score_floor=0.05, nms_iou=0.45, threads=4,
    # palmprint
    roi=128, code_stride=4, gabor_ksize=17, gabor_sigma=3.5, gabor_lambda=9.0, shift=2, min_palm_side=96,
    enroll_frames=40, gallery_templates=10,              # 10 templates + 30 calibration (≥ 29, Lean §3)
    id_margin=0.06,                                      # τ_person must sit ≥ this far below chance-level distance
    debounce_N=6, debounce_K=3, live_seconds=60, port=8361, run_lean=True,
    lean_url='https://github.com/leanprover/lean4/releases/download/v4.31.0/lean-4.31.0-linux.tar.zst',
    yolox_repo='https://github.com/Megvii-BaseDetection/YOLOX.git',
    yolox_weights='https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth',
    yunet_url='https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx',
    palm_url='https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/palm_detection_mediapipe/palm_detection_mediapipe_2023feb.onnx',
)
REPORT = {}

# ── §0 deps ─────────────────────────────────────────────────────────────────────
def _need(mod, pip=None):
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip or mod])
for m in ['onnx', 'onnxruntime', 'scipy', 'loguru', 'fastapi', 'uvicorn', 'zstandard', 'torchvision', 'onnxscript', 'tabulate']:
    _need(m)
YOLOX_DIR = os.environ.get('VG360_YOLOX', os.path.join(WORK, 'YOLOX'))
if not os.path.isdir(os.path.join(YOLOX_DIR, 'yolox')):
    subprocess.check_call(['git', 'clone', '-q', '--depth', '1', CFG['yolox_repo'], YOLOX_DIR])
sys.path.insert(0, YOLOX_DIR)
import numpy as np, cv2, torch, torch.nn as nn, onnxruntime as ort
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
np.random.seed(0); torch.manual_seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; NC, S = len(CFG['classes']), CFG['input_size']

def sanitize(o):
    if isinstance(o, dict): return {str(k): sanitize(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [sanitize(v) for v in o]
    if isinstance(o, torch.Tensor): return sanitize(o.detach().cpu().numpy())
    if isinstance(o, np.ndarray): return sanitize(o.tolist())
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.floating, float)): f = float(o); return f if math.isfinite(f) else None
    return o

def fetch(url, name):
    p = os.environ.get('VG360_' + name.split('.')[0].upper(), os.path.join(WORK, name))
    if not os.path.exists(p): urllib.request.urlretrieve(url, p)
    return p

# ── §1 Lean 4.31 proofs (kernel-only) ────────────────────────────────────────────
LEAN_SRC = r'''/-
VG360 mono face/palm + palmprint ID: formal guarantees (Lean 4.31, kernel-only: Nat + omega, no Mathlib).
CompCode: each code cell holds a dominant line orientation o ∈ {0..5} (6 Gabor filters, 30° apart).
-/

-- §1 Angular distance between two orientation codes on the 6-cycle.
def absDiff (a b : Nat) : Nat := if a ≤ b then b - a else a - b
def angDist (a b : Nat) : Nat := if absDiff a b ≤ 3 then absDiff a b else 6 - absDiff a b

theorem absDiff_symm (a b : Nat) : absDiff a b = absDiff b a := by
  unfold absDiff; split <;> split <;> omega

theorem angDist_symm (a b : Nat) : angDist a b = angDist b a := by
  unfold angDist; rw [absDiff_symm a b]

theorem angDist_self (a : Nat) : angDist a a = 0 := by
  unfold angDist absDiff; simp

/-- Per-cell distance never exceeds 3 for valid codes, so a template of n cells has total ≤ 3n:
    normalising by 3n gives a score in [0,1]. -/
theorem angDist_le_three (a b : Nat) (ha : a < 6) (hb : b < 6) : angDist a b ≤ 3 := by
  have : absDiff a b ≤ 5 := by unfold absDiff; split <;> omega
  unfold angDist; split <;> omega

def total : List (Nat × Nat) → Nat
  | [] => 0
  | (a, b) :: t => angDist a b + total t

theorem total_le (l : List (Nat × Nat)) (h : ∀ p ∈ l, p.1 < 6 ∧ p.2 < 6) : total l ≤ 3 * l.length := by
  induction l with
  | nil => simp [total]
  | cons p t ih =>
    have hp := h p (by simp)
    have ht : ∀ q ∈ t, q.1 < 6 ∧ q.2 < 6 := fun q hq => h q (by simp [hq])
    have := angDist_le_three p.1 p.2 hp.1 hp.2
    have := ih ht
    simp [total, List.length]; omega

-- §2 Open-set accept decision (scaled integers: dist·D ≤ τ·D, cross-multiplied, no division).
def accept (dist tau : Nat) : Prop := dist ≤ tau

/-- A looser threshold never rejects a probe that a stricter one accepted. -/
theorem accept_monotone (d t t' : Nat) (h : t ≤ t') (ha : accept d t) : accept d t' := by
  unfold accept at *; omega

/-- Best-of-gallery matching: if any template is accepted, the minimum distance is accepted. -/
theorem min_accept (d₁ d₂ t : Nat) (h : accept d₁ t ∨ accept d₂ t) : accept (min d₁ d₂) t := by
  unfold accept at *; omega

/-- Identity disambiguation uses dist·τ_other < dist_other·τ (ratio comparison, division-free):
    it is irreflexive, so a probe is never ranked strictly closer to a person than to itself. -/
def closerRel (d t d' t' : Nat) : Prop := d * t' < d' * t
theorem closerRel_irrefl (d t : Nat) : ¬ closerRel d t d t := by
  unfold closerRel; omega

-- §3 Mondrian split-conformal per identity / per class at α = 1/30:
--     rank k = ⌈(n+1)(1−α)⌉ = ((n+1)·29 + 29)/30 must satisfy k ≤ n.
def confRank (n : Nat) : Nat := ((n + 1) * 29 + 29) / 30
theorem conformal_feasible (n : Nat) (h : 29 ≤ n) : confRank n ≤ n := by
  unfold confRank; omega
theorem conformal_tight : confRank 28 > 28 := by decide

-- §4 K-of-N debounce for identity announcements (saturating counter).
def step (N cnt : Nat) (hit : Bool) : Nat := if hit then min (cnt + 1) N else cnt - 1
theorem step_bounded (N cnt : Nat) (hit : Bool) (h : cnt ≤ N) : step N cnt hit ≤ N := by
  unfold step; cases hit <;> simp <;> omega
theorem step_lipschitz (N cnt : Nat) (hit : Bool) : step N cnt hit ≤ cnt + 1 := by
  unfold step; cases hit <;> simp <;> omega
def run (N : Nat) : Nat → List Bool → Nat
  | c, [] => c
  | c, b :: bs => run N (step N c b) bs
theorem run_le (N : Nat) : ∀ (c : Nat) (bs : List Bool), run N c bs ≤ c + bs.length
  | c, [] => by simp [run]
  | c, b :: bs => by
      have h1 := step_lipschitz N c b
      have h2 := run_le N (step N c b) bs
      simp [run, List.length]; omega
theorem announce_needs_K_frames (N K : Nat) (bs : List Bool) (h : K ≤ run N 0 bs) : K ≤ bs.length := by
  have := run_le N 0 bs; omega

-- §5 Resolution gate: palm-line codes are computed only if the palm box side ≥ minSide px;
--     code grid cells = side / stride, so the gate guarantees ≥ minSide/stride cells per axis.
theorem gate_cells (side minSide stride : Nat) (h : minSide ≤ side) :
    minSide / stride ≤ side / stride := Nat.div_le_div_right h
'''
LEAN_THMS = ['absDiff_symm', 'angDist_symm', 'angDist_self', 'angDist_le_three', 'total_le', 'accept_monotone',
             'min_accept', 'closerRel_irrefl', 'conformal_feasible', 'conformal_tight', 'step_bounded',
             'step_lipschitz', 'run_le', 'announce_needs_K_frames', 'gate_cells']
def lean_verify():
    lean = shutil.which('lean') or os.environ.get('VG360_LEAN')
    if not lean:
        lean = os.path.join(WORK, 'lean-4.31.0-linux', 'bin', 'lean')
        if not os.path.exists(lean):
            import zstandard, tarfile; arc = os.path.join(WORK, 'lean.tar.zst')
            print('[lean] downloading Lean 4.31.0 (~560 MB) …'); urllib.request.urlretrieve(CFG['lean_url'], arc)
            with open(arc, 'rb') as f: tarfile.open(fileobj=zstandard.ZstdDecompressor().stream_reader(f), mode='r|').extractall(WORK)
            os.remove(arc)
    path = os.path.join(WORK, 'PalmID.lean')
    open(path, 'w').write(LEAN_SRC + '\n' + '\n'.join(f'#print axioms {t}' for t in LEAN_THMS) + '\n')
    r = subprocess.run([lean, path], capture_output=True, text=True, timeout=600); out = r.stdout + r.stderr
    ok = r.returncode == 0 and 'sorryAx' not in out and 'error' not in out.lower()
    return dict(ok=ok, lean=subprocess.run([lean, '--version'], capture_output=True, text=True).stdout.strip(),
                theorems=len(LEAN_THMS), axioms=sorted(set(re.findall(r'\b(propext|Classical\.choice|Quot\.sound|sorryAx)\b', out))),
                log='' if ok else out[-2000:])
if CFG['run_lean']:
    REPORT['lean'] = lean_verify(); print('[lean]', 'VERIFIED' if REPORT['lean']['ok'] else 'FAILED', REPORT['lean']['lean'])
    assert REPORT['lean']['ok'], REPORT['lean']['log']
# integer mirrors of the proven definitions
def lean_angDist(a, b): d = np.abs(a.astype(np.int16) - b.astype(np.int16)); return np.where(d <= 3, d, 6 - d)
def lean_confRank(n): return ((n + 1) * 29 + 29) // 30
def lean_step(N, c, hit): return min(c + 1, N) if hit else max(c - 1, 0)

# ── §2 camera: Colab browser webcam · local cv2 webcam · file (video/gif/images) ──
class Camera:
    def __init__(self, src):
        self.src, self.kind = src, None
        if src == 'webcam' and IN_COLAB:
            from IPython.display import display, Javascript
            from google.colab import output; self.output = output
            display(Javascript('''
              window.vgInit = async () => { if (window.vgV) return true; const v = document.createElement('video');
                v.style.display='none'; document.body.appendChild(v);
                window.vgS = await navigator.mediaDevices.getUserMedia({video:{width:%d,height:%d}});
                v.srcObject = window.vgS; await v.play(); window.vgV = v; const c = document.createElement('canvas');
                c.width = v.videoWidth; c.height = v.videoHeight; window.vgC = c; return true; };
              window.vgGrabN = async (n, ms) => { const out = []; const ctx = window.vgC.getContext('2d');
                for (let i = 0; i < n; i++) { ctx.drawImage(window.vgV,0,0); out.push(window.vgC.toDataURL('image/jpeg',0.8));
                  if (i < n - 1) await new Promise(r => setTimeout(r, ms)); } return out; };
              window.vgStop = () => { if (window.vgS) { window.vgS.getTracks().forEach(t=>t.stop()); window.vgV.remove(); window.vgV=null; } return true; };''' % (CFG['cam_w'], CFG['cam_h'])))
            print('[camera] allow the browser camera prompt if it appears …'); t_ = time.time()
            output.eval_js('vgInit()', timeout_sec=60); self.kind, self.buf = 'colab', []
            print(f'[camera] browser webcam ready ({time.time() - t_:.1f}s)')
        elif src == 'webcam':
            self.cap = cv2.VideoCapture(0); self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, CFG['cam_w']); self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CFG['cam_h'])
            assert self.cap.isOpened(), 'no local webcam'; self.kind = 'cv2'
        elif any(ch in src for ch in '*?'):
            self.files, self.i, self.kind = sorted(glob.glob(src)), 0, 'files'
        else:
            self.cap, self.kind = cv2.VideoCapture(src), 'cv2file'
    def read(self):
        """Returns a COLOUR BGR frame (colour is only used by the teachers; the runtime path is mono)."""
        if self.kind == 'colab':
            if not self.buf:   # one JS round trip fetches grab_batch frames (round trip dominates in Colab)
                urls = self.output.eval_js(f"vgGrabN({CFG['grab_batch']}, {CFG['grab_ms']})", timeout_sec=30)
                self.buf = [cv2.imdecode(np.frombuffer(base64.b64decode(u.split(',', 1)[1]), np.uint8), cv2.IMREAD_COLOR) for u in urls]
            return self.buf.pop(0)
        if self.kind == 'files':
            if self.i >= len(self.files): return None
            self.i += 1; return cv2.imread(self.files[self.i - 1])
        ok, f = self.cap.read()
        if not ok and self.kind == 'cv2file': self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0); ok, f = self.cap.read()
        return f if ok else None
    def close(self):
        if self.kind == 'colab': self.output.eval_js('vgStop()')
        elif hasattr(self, 'cap'): self.cap.release()

class Viewer:
    def __init__(self):
        self.h = None
    def show(self, img):
        if IN_COLAB or 'ipykernel' in sys.modules:
            from IPython.display import display, Image as IPImage
            data = cv2.imencode('.jpg', img)[1].tobytes()
            if self.h is None: self.h = display(IPImage(data=data), display_id=True)
            else: self.h.update(IPImage(data=data))
        elif os.environ.get('DISPLAY'): cv2.imshow('VG360-PALMID', img); cv2.waitKey(1)

# ── §3 colour teachers ─────────────────────────────────────────────────────────
YUNET = cv2.FaceDetectorYN.create(fetch(CFG['yunet_url'], 'yunet.onnx'), '', (320, 320), CFG['teacher_face_thr'], 0.3, 5000)
PALM_T = ort.InferenceSession(fetch(CFG['palm_url'], 'palm.onnx'), providers=['CPUExecutionProvider'])
_PA = []
for n_, k_ in ((24, 2), (12, 6)):                       # 2016 SSD anchors (verified vs opencv_zoo table, max err 3e-8)
    for y in range(n_):
        for x in range(n_): _PA += [[(x + .5) / n_, (y + .5) / n_]] * k_
_PA = np.array(_PA, np.float32)

def teacher_faces(bgr):
    YUNET.setInputSize((bgr.shape[1], bgr.shape[0])); _, f = YUNET.detect(bgr)
    if f is None: return np.zeros((0, 4), np.float32)
    return np.stack([f[:, 0], f[:, 1], f[:, 0] + f[:, 2], f[:, 1] + f[:, 3]], 1).astype(np.float32)

def teacher_palms(bgr, thr=None):
    thr = thr or CFG['teacher_palm_thr']; h, w = bgr.shape[:2]; r = 192 / max(h, w); nh, nw = int(h * r), int(w * r)
    top, left = (192 - nh) // 2, (192 - nw) // 2
    img = cv2.copyMakeBorder(cv2.resize(bgr, (nw, nh)), top, 192 - nh - top, left, 192 - nw - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    reg, sc = PALM_T.run(None, {'input_1': (img[..., ::-1].astype(np.float32) / 255.0)[None]})
    reg, sc = reg[0], 1 / (1 + np.exp(-sc[0, :, 0].astype(np.float64)))
    c = reg[:, :2] / 192 + _PA; wh = reg[:, 2:4] / 192
    b = (np.concatenate([c - wh / 2, c + wh / 2], 1) * 192 - [left, top, left, top]) / r
    keep = np.array(cv2.dnn.NMSBoxes([[*q[:2], *(q[2:] - q[:2])] for q in b], sc.tolist(), thr, 0.3), np.int64).reshape(-1)
    b = b[keep]; b[:, [0, 2]] = b[:, [0, 2]].clip(0, w - 1); b[:, [1, 3]] = b[:, [1, 3]].clip(0, h - 1)
    return b.astype(np.float32)

# ── §4 collect teacher-labelled frames, store MONO copies (YOLO format) ──────────
DATA = os.path.join(WORK, 'data'); shutil.rmtree(DATA, ignore_errors=True)
print('[camera] opening', CFG['source'], '…'); cam = Camera(CFG['source']); frames = []; gray_test = dict(colour_palms=0, gray_palms=0, frames_checked=0)
print(f"[collect] up to {CFG['collect_frames']} frames / {CFG['collect_seconds']} s — show your face and OPEN PALMS, vary distance and angle …")
k, t_c, view_c, nf_tot, np_tot = 0, time.time(), Viewer(), 0, 0
while len(frames) < CFG['collect_frames'] and time.time() - t_c < CFG['collect_seconds']:
    t_r = time.time(); f = cam.read()
    if f is None: print('[collect] camera returned no frame — stopping'); break
    if time.time() - t_r > 20: print(f'[collect] slow camera read ({time.time() - t_r:.0f}s)')
    k += 1
    if k % CFG['collect_every']: continue
    fb, pb = teacher_faces(f), teacher_palms(f); g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY); nf_tot += len(fb); np_tot += len(pb)
    if len(frames) % 10 == 0:          # sparse colour-vs-gray teacher diagnostic (doubles palm cost, so not every frame)
        gray_test['frames_checked'] += 1; gray_test['colour_palms'] += len(pb)
        gray_test['gray_palms'] += len(teacher_palms(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)))
    frames.append((g, np.concatenate([fb, pb]), np.r_[np.zeros(len(fb)), np.ones(len(pb))].astype(np.int64)))
    if len(frames) % CFG['preview_every'] == 0:
        vis = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
        for q in fb.astype(int): cv2.rectangle(vis, tuple(q[:2]), tuple(q[2:]), (255, 255, 255), 2)
        for q in pb.astype(int): cv2.rectangle(vis, tuple(q[:2]), tuple(q[2:]), (0, 255, 255), 2)
        cv2.putText(vis, f'collect {len(frames)}/{CFG["collect_frames"]}  faces {nf_tot}  palms {np_tot}', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        view_c.show(vis)
    if len(frames) % CFG['progress_every'] == 0:
        el = time.time() - t_c
        print(f'[collect] {len(frames)}/{CFG["collect_frames"]}  {len(frames) / el:.1f} fps  faces={nf_tot} palms={np_tot}  {el:.0f}s')
assert len(frames) >= 20, 'too few frames captured — check camera permission / source'
if np_tot < 30: print(f'[collect] WARNING only {np_tot} palm labels — the student will learn palms poorly; re-run and show open palms more')
REPORT['teacher_mono_gap'] = gray_test
print('[collect]', len(frames), 'frames | teacher palms colour vs gray:', gray_test)
n = len(frames); cut1, cut2 = int(n * 0.6), int(n * 0.8)                  # time-ordered split, no leakage
SPLITS = {'train': frames[:cut1], 'cal': frames[cut1:cut2], 'val': frames[cut2:]}
for s_, items in SPLITS.items():
    os.makedirs(f'{DATA}/images/{s_}', exist_ok=True); os.makedirs(f'{DATA}/labels/{s_}', exist_ok=True)
    for i, (g, b, c) in enumerate(items):
        cv2.imwrite(f'{DATA}/images/{s_}/{i:05d}.png', g); H, W = g.shape
        with open(f'{DATA}/labels/{s_}/{i:05d}.txt', 'w') as fh:
            for (x1, y1, x2, y2), ci in zip(b, c): fh.write(f'{ci} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}\n')

# ── §5 mono student: YOLOX-Nano trained with SAM ──────────────────────────────────
_CLAHE = cv2.createCLAHE(2.0, (8, 8))
def letterbox(gray):
    H, W = gray.shape; r = min(S / H, S / W); g = cv2.resize(_CLAHE.apply(gray), (int(W * r), int(H * r)))
    pad = np.full((S, S), 114, np.uint8); pad[:g.shape[0], :g.shape[1]] = g
    return np.repeat(pad[None], 3, 0).astype(np.float32), r

def mono_augment(g, b, c, rng):                     # intensity/texture/geometry only — no HSV
    H, W = g.shape; g = np.clip(g.astype(np.float32) * rng.uniform(0.6, 1.4) + rng.uniform(-30, 30), 0, 255)
    if rng.random() < 0.3: g = 255 * (g / 255) ** rng.uniform(0.6, 1.6)
    g = g.astype(np.uint8)
    if rng.random() < 0.3: g = cv2.GaussianBlur(g, (0, 0), rng.uniform(0.5, 1.5))
    if rng.random() < 0.3: g = np.clip(g + rng.normal(0, rng.uniform(3, 10), g.shape), 0, 255).astype(np.uint8)
    s, tx, ty = rng.uniform(0.7, 1.3), rng.uniform(-.1, .1) * W, rng.uniform(-.1, .1) * H
    M = np.float32([[s, 0, tx + (1 - s) * W / 2], [0, s, ty + (1 - s) * H / 2]]); g = cv2.warpAffine(g, M, (W, H), borderValue=114)
    if len(b):
        b = b * s; b[:, [0, 2]] += M[0, 2]; b[:, [1, 3]] += M[1, 2]
        b[:, [0, 2]] = b[:, [0, 2]].clip(0, W - 1); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H - 1)
        keep = ((b[:, 2] - b[:, 0]) > 4) & ((b[:, 3] - b[:, 1]) > 4); b, c = b[keep], c[keep]
    if rng.random() < 0.5:
        g = g[:, ::-1].copy()
        if len(b): b = b.copy(); b[:, [0, 2]] = W - 1 - b[:, [2, 0]]
    return g, b, c

def make_batch(items, rng, aug):
    X = np.zeros((len(items), 3, S, S), np.float32); T = np.zeros((len(items), 30, 5), np.float32)
    for i, (g, b, c) in enumerate(items):
        if aug: g, b, c = mono_augment(g, b.copy(), c.copy(), rng)
        X[i], r = letterbox(g); b = b * r; m = min(len(b), 30)
        if m: T[i, :m] = np.c_[c[:m], (b[:m, 0] + b[:m, 2]) / 2, (b[:m, 1] + b[:m, 3]) / 2, b[:m, 2] - b[:m, 0], b[:m, 3] - b[:m, 1]]
    return torch.from_numpy(X), torch.from_numpy(T)

def build_nano():
    inch = [256, 512, 1024]
    m = YOLOX(YOLOPAFPN(0.33, 0.25, in_channels=inch, depthwise=True), YOLOXHead(NC, 0.25, in_channels=inch, depthwise=True))
    for x in m.modules():
        if isinstance(x, nn.BatchNorm2d): x.eps, x.momentum = 1e-3, 0.03
    m.head.initialize_biases(1e-2); return m
model = build_nano()
ck = torch.load(fetch(CFG['yolox_weights'], 'yolox_nano.pth'), map_location='cpu', weights_only=False)['model']; sd = model.state_dict()
model.load_state_dict({k: v for k, v in ck.items() if k in sd and v.shape == sd[k].shape}, strict=False); model.to(DEV)

class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimisation (Foret et al. 2021)."""
    def __init__(self, params, base, rho=0.05, adaptive=False, **kw):
        super().__init__(params, dict(rho=rho, adaptive=adaptive, **kw)); self.base_optimizer = base(self.param_groups, **kw)
        self.param_groups = self.base_optimizer.param_groups; self.defaults.update(self.base_optimizer.defaults)
    @torch.no_grad()
    def first_step(self):
        gn = torch.norm(torch.stack([((torch.abs(p) if g['adaptive'] else 1.0) * p.grad).norm(2)
                                     for g in self.param_groups for p in g['params'] if p.grad is not None]), 2)
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is None: continue
                self.state[p]['old_p'] = p.data.clone()
                p.add_((torch.pow(p, 2) if g['adaptive'] else 1.0) * p.grad * (g['rho'] / (gn + 1e-12)))
    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is not None: p.data = self.state[p]['old_p']
        self.base_optimizer.step()
class bn_frozen:
    def __init__(self, m): self.bn = [x for x in m.modules() if isinstance(x, nn.modules.batchnorm._BatchNorm)]
    def __enter__(self): self.mo = [b.momentum for b in self.bn]; [setattr(b, 'momentum', 0.0) for b in self.bn]
    def __exit__(self, *a): [setattr(b, 'momentum', m) for b, m in zip(self.bn, self.mo)]

dec = [p for p in model.parameters() if p.ndim > 1]; nod = [p for p in model.parameters() if p.ndim <= 1]
opt = SAM([dict(params=dec, weight_decay=CFG['wd']), dict(params=nod, weight_decay=0.0)], torch.optim.SGD,
          rho=CFG['sam_rho'], adaptive=CFG['sam_adaptive'], lr=CFG['lr'], momentum=CFG['momentum'], nesterov=True)
ema = build_nano().to(DEV); ema.load_state_dict(model.state_dict()); ema.eval()
tr = SPLITS['train']; ipe = max(1, len(tr) // CFG['batch']); total = ipe * CFG['epochs']; it = 0; rng = np.random.default_rng(1); t0 = time.time()
for ep in range(CFG['epochs']):
    model.train(); idx = rng.permutation(len(tr)); L = []
    for bi in range(ipe):
        X, T = make_batch([tr[j] for j in idx[bi * CFG['batch']:(bi + 1) * CFG['batch']]], rng, True); X, T = X.to(DEV), T.to(DEV)
        lr = CFG['lr'] * min(1, (it / CFG['warmup_iters']) ** 2) if it < CFG['warmup_iters'] else \
             CFG['lr'] * (0.05 + 0.475 * (1 + math.cos(math.pi * (it - CFG['warmup_iters']) / max(1, total - CFG['warmup_iters']))))
        for g in opt.param_groups: g['lr'] = lr
        loss = model(X, T)['total_loss']; opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 10); opt.first_step(); opt.zero_grad()
        with bn_frozen(model): model(X, T)['total_loss'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 10); opt.second_step(); opt.zero_grad()
        with torch.no_grad():
            d = min(0.9998, (1 + it) / (10 + it)); msd = model.state_dict()
            for k_, v in ema.state_dict().items():
                if v.dtype.is_floating_point: v.mul_(d).add_(msd[k_].detach(), alpha=1 - d)
                else: v.copy_(msd[k_])
        L.append(float(loss)); it += 1
    print(f'[train] epoch {ep+1}/{CFG["epochs"]} loss={np.mean(L):.3f} lr={lr:.5f} {time.time()-t0:.0f}s')
REPORT['train'] = dict(optimizer='SAM(SGD-nesterov)', rho=CFG['sam_rho'], iters=it, final_loss=np.mean(L), seconds=time.time() - t0)

# ── §6 decode · INT8 export · Mondrian SCCP thresholds · MCC vs teacher ─────────────
_G, _ST = [], []
for st in (8, 16, 32):
    h_ = S // st; ys, xs = np.meshgrid(np.arange(h_), np.arange(h_), indexing='ij')
    _G.append(np.stack([xs, ys], -1).reshape(-1, 2)); _ST.append(np.full((h_ * h_, 1), st))
_G, _ST = np.concatenate(_G).astype(np.float32), np.concatenate(_ST).astype(np.float32)
def decode(raw, r, tau):
    xy = (raw[:, :2] + _G) * _ST; wh = np.exp(np.clip(raw[:, 2:4], -10, 10)) * _ST
    cls = raw[:, 5:].argmax(1); sc = raw[:, 4] * raw[np.arange(len(raw)), 5 + cls]; m = sc >= np.asarray(tau)[cls]
    b = (np.concatenate([xy - wh / 2, xy + wh / 2], 1) / r)[m]; sc, cls = sc[m], cls[m]; ob, os_, oc = [], [], []
    for c in np.unique(cls):
        k = np.where(cls == c)[0]
        keep = np.array(cv2.dnn.NMSBoxes([[*q[:2], *(q[2:] - q[:2])] for q in b[k]], sc[k].tolist(), 0.0, CFG['nms_iou']), np.int64).reshape(-1)
        ob.append(b[k][keep]); os_.append(sc[k][keep]); oc.append(cls[k][keep])
    if not ob: return np.zeros((0, 4), np.float32), np.zeros(0), np.zeros(0, np.int64)
    return np.concatenate(ob), np.concatenate(os_), np.concatenate(oc)

ema_cpu = build_nano(); ema_cpu.load_state_dict(ema.state_dict()); ema_cpu.eval(); ema_cpu.head.decode_in_inference = False
F32, PRE, I8 = (os.path.join(WORK, f) for f in ('palmid_fp32.onnx', 'palmid_pre.onnx', 'palmid_int8.onnx'))
try: torch.onnx.export(ema_cpu, torch.zeros(1, 3, S, S), F32, input_names=['images'], output_names=['output'], opset_version=17, dynamo=False)
except TypeError: torch.onnx.export(ema_cpu, torch.zeros(1, 3, S, S), F32, input_names=['images'], output_names=['output'], opset_version=17)
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
quant_pre_process(F32, PRE, skip_symbolic_shape=True)
class _Cal(CalibrationDataReader):
    def __init__(s, it): s.it = iter([letterbox(g)[0][None] for g, _, _ in it])
    def get_next(s): x = next(s.it, None); return None if x is None else {'images': x}
quantize_static(PRE, I8, _Cal(SPLITS['cal'][:64]), quant_format=QuantFormat.QDQ, per_channel=True,
                weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)
so = ort.SessionOptions(); so.intra_op_num_threads = CFG['threads']
STU = ort.InferenceSession(I8, so, providers=['CPUExecutionProvider'])
def student(gray, tau):
    x, r = letterbox(gray); return decode(STU.run(None, {'images': x[None]})[0][0], r, tau)

def iou(a, b):
    if not len(a) or not len(b): return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    i_ = (x2 - x1).clip(0) * (y2 - y1).clip(0); aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return i_ / (aa[:, None] + bb[None] - i_ + 1e-9)
def mcc(tp, fp, fn, tn):
    d = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)); return (tp * tn - fp * fn) / d if d else 0.0

floor = [CFG['score_floor']] * NC
cal_out = [(b, c, *student(g, floor)) for g, b, c in SPLITS['cal']]
TAU, TAU_INFO = [], {}
for c in range(NC):
    ncs = []
    for gb, gc, pb, ps, pc in cal_out:
        M = iou(gb[gc == c], pb[pc == c]); P = ps[pc == c]
        for gi in range(M.shape[0]): ok = M[gi] >= .5; ncs.append(1 - (P[ok].max() if ok.any() else 0.0))
    n_ = len(ncs)
    if n_ >= CFG['n_cal_bound']:
        q = 1 - sorted(ncs)[lean_confRank(n_) - 1]; t = max(CFG['score_floor'], q)
        src = 'sccp' if q >= CFG['score_floor'] else 'sccp_floor(coverage not attainable)'
    else: t, src = CFG['tau_fallback'], f'fallback(n={n_}<29)'
    TAU.append(float(t)); TAU_INFO[CFG['classes'][c]] = dict(n_cal=n_, tau=t, source=src)
def eval_mcc(items, tau):
    res = {}
    for c in range(NC):
        tp = fp = fn = tn = 0
        for g, b, cl in items:
            _, _, pc = student(g, tau); y, yh = (cl == c).any(), (pc == c).any()
            tp += y and yh; fp += (not y) and yh; fn += y and not yh; tn += (not y) and not yh
        res[CFG['classes'][c]] = dict(MCC=round(mcc(tp, fp, fn, tn), 4), TP=tp, FP=fp, FN=fn, TN=tn)
    return res
REPORT['sccp'] = TAU_INFO; REPORT['mcc_student_int8_vs_teacher'] = eval_mcc(SPLITS['val'], TAU)
# sidecar for the Rust runtime (github.com/dhaneshkk/test1_palm): palmid --model palmid_int8.onnx --meta palmid_model.json
json.dump(dict(classes=CFG['classes'], input_size=S, tau=[float(t) for t in TAU], score_floor=CFG['score_floor'], nms_iou=CFG['nms_iou']),
          open(os.path.join(WORK, 'palmid_model.json'), 'w'), indent=1)
print('[export] Rust runtime files:', I8, os.path.join(WORK, 'palmid_model.json'))
print('[eval] τ:', {k: round(v['tau'], 3) for k, v in TAU_INFO.items()}, '| MCC (INT8 student vs colour teacher, val):',
      {k: v['MCC'] for k, v in REPORT['mcc_student_int8_vs_teacher'].items()})

# ── §7 palmprint: ROI → CompCode → open-set identification ──────────────────────
def _gabor_bank():
    ks = []
    for j in range(6):
        k = cv2.getGaborKernel((CFG['gabor_ksize'],) * 2, CFG['gabor_sigma'], j * np.pi / 6, CFG['gabor_lambda'], 0.7, 0, ktype=cv2.CV_32F)
        ks.append(k - k.mean())                      # zero-DC: illumination-invariant
    return ks
GABOR = _gabor_bank()

def palm_roi(gray, box):
    """Central 70 % square of the palm box (excludes fingers/background). None if below resolution gate."""
    x1, y1, x2, y2 = box; side = min(x2 - x1, y2 - y1)
    if side < CFG['min_palm_side']: return None                                   # Lean §5 gate
    cx, cy, half = (x1 + x2) / 2, (y1 + y2) / 2, 0.35 * side
    a, b_, c_, d_ = int(cx - half), int(cy - half), int(cx + half), int(cy + half)
    if a < 0 or b_ < 0 or c_ > gray.shape[1] or d_ > gray.shape[0]: return None
    return cv2.resize(_CLAHE.apply(gray[b_:d_, a:c_]), (CFG['roi'], CFG['roi']), interpolation=cv2.INTER_AREA)

def compcode(roi):
    """Competitive code: per cell, orientation of the strongest dark-line (min) Gabor response ∈ {0..5}."""
    f = cv2.GaussianBlur(roi, (0, 0), 1.0).astype(np.float32)   # suppress sensor/compression noise
    R = np.stack([cv2.filter2D(f, -1, k) for k in GABOR]); code = R.argmin(0).astype(np.uint8)
    st = CFG['code_stride']; m = CFG['gabor_ksize'] // 2
    return code[m:-m:st, m:-m:st]

def code_dist(a, b):
    """Normalised angular distance in [0,1] (Lean §1: per-cell ≤ 3 ⇒ total ≤ 3n), min over ±shift translations."""
    s, best = CFG['shift'], 1.0
    for dy in range(-s, s + 1):
        for dx in range(-s, s + 1):
            A = a[max(0, dy):a.shape[0] + min(0, dy), max(0, dx):a.shape[1] + min(0, dx)]
            B = b[max(0, -dy):b.shape[0] + min(0, -dy), max(0, -dx):b.shape[1] + min(0, -dx)]
            best = min(best, float(lean_angDist(A, B).sum()) / (3 * A.size))
    return best

GALLERY_PATH = os.path.join(WORK, 'palm_gallery.npz')
class PalmGallery:
    """Enrolled identities only. Stores CompCode templates (3-bit orientation codes) + per-person τ. No images."""
    def __init__(self):
        self.g = {}
        if os.path.exists(GALLERY_PATH):
            z = np.load(GALLERY_PATH, allow_pickle=True); self.g = z['g'].item()
    def save(self): np.savez(GALLERY_PATH, g=np.array(self.g, dtype=object))
    def enroll(self, name, codes, consent):
        assert consent is True, 'enrollment requires the person’s explicit consent'
        G, C = codes[:CFG['gallery_templates']], codes[CFG['gallery_templates']:]
        genuine = sorted(min(code_dist(c, t) for t in G) for c in C); n_ = len(genuine)
        if n_ >= CFG['n_cal_bound']: tau, src = genuine[lean_confRank(n_) - 1], 'sccp'          # Lean §3
        else: tau, src = (genuine[-1] if genuine else 0.30), f'max-genuine(n={n_}<29, no guarantee)'
        # chance level: distance to cell-shuffled templates (same code statistics, no palm structure)
        rs = np.random.default_rng(0)
        chance = float(np.median([code_dist(c, rs.permutation(G[i % len(G)].ravel()).reshape(G[0].shape)) for i, c in enumerate(C[:10] or G)]))
        info = dict(tau=float(tau), chance=chance, n_cal=n_, source=src, genuine_median=float(np.median(genuine)) if genuine else None)
        if tau > chance - CFG['id_margin']:
            # codes are indistinguishable from noise → a threshold here would accept everyone. Refuse.
            return dict(enrolled=False, reason='palm texture not resolved (move closer / more light / sharper focus)', **info)
        self.g[name] = dict(templates=G, **info); self.save()
        return dict(enrolled=True, **info)
    def delete(self, name): self.g.pop(name, None); self.save()
    def identify(self, code):
        """Open-set: accept person p iff d_p ≤ τ_p; among accepted, smallest d_p/τ_p wins; else 'unknown'."""
        best, cand = None, []
        for name, e in self.g.items():
            d = min(code_dist(code, t) for t in e['templates']); cand.append((name, d, e['tau']))
            if d <= e['tau'] and (best is None or d * best[2] < best[1] * e['tau']): best = (name, d, e['tau'])
        return (best[0] if best else 'unknown'), cand

GAL = PalmGallery()
def capture_palm_codes(cam, n_codes, max_frames=2000, sel=None):
    codes, seen = [], 0
    while len(codes) < n_codes and seen < max_frames:
        f = cam.read(); seen += 1
        if f is None: break
        g = f if f.ndim == 2 else cv2.cvtColor(f, cv2.COLOR_BGR2GRAY); b, s, c = student(g, TAU)
        palms = b[c == 1]
        if sel is not None: palms = palms[[sel(q, g.shape) for q in palms]] if len(palms) else palms
        if len(palms):
            roi = palm_roi(g, palms[np.argmax((palms[:, 2] - palms[:, 0]) * (palms[:, 3] - palms[:, 1]))])
            if roi is not None: codes.append(compcode(roi))
    return codes

# ── §8 enrollment + live loop ──────────────────────────────────────────────────
def enroll_interactive():
    while True:
        name = input('Enroll a person? Enter name (blank to stop): ').strip()
        if not name: break
        ok = input(f'Does {name} consent to palmprint enrollment on this machine? (type YES): ').strip() == 'YES'
        if not ok: print('skipped — no consent'); continue
        print(f'{name}: hold ONE open palm 30–50 cm from the camera, fingers spread, move slowly …')
        codes = capture_palm_codes(cam, CFG['enroll_frames'])
        if len(codes) < CFG['gallery_templates'] + 1: print('not enough palm captures; retry closer / better light'); continue
        print('[enroll]', name, GAL.enroll(name, codes, consent=True))

def draw_and_identify(gray, deb):
    b, s, c = student(gray, TAU); vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR); out = []
    for q, sc, ci in zip(b, s, c):
        x1, y1, x2, y2 = q.astype(int); col = (255, 255, 255) if ci == 0 else (0, 255, 255)
        label = f"{CFG['classes'][ci]} {sc:.2f}"; rec = dict(cls=CFG['classes'][ci], score=sc, box=q)
        if ci == 1:
            roi = palm_roi(gray, q)
            if roi is None: label += ' (move closer)'
            else:
                who, cand = GAL.identify(compcode(roi)); key = ('id', who)
                deb[key] = lean_step(CFG['debounce_N'], deb.get(key, 0), True)
                for k_ in list(deb):
                    if k_ != key: deb[k_] = lean_step(CFG['debounce_N'], deb[k_], False)
                stable = deb[key] >= CFG['debounce_K']                                    # Lean §4
                rec.update(identity=who if stable else 'pending', candidates=cand); label += f' → {who if stable else "…"}'
        cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2); cv2.putText(vis, label, (x1, max(15, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        out.append(rec)
    return vis, out

if CFG['source'] == 'webcam':
    enroll_interactive()
    view, deb, t_end = Viewer(), {}, time.time() + CFG['live_seconds']; nfr = 0; t0 = time.time()
    while time.time() < t_end:
        f = cam.read()
        if f is None: break
        vis, _ = draw_and_identify(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY), deb); nfr += 1
        cv2.putText(vis, f'{nfr / (time.time() - t0):.1f} fps  MONO', (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2); view.show(vis)
    REPORT['live'] = dict(frames=nfr, fps=nfr / max(1e-6, time.time() - t0))
else:
    # file-source self-test: the two hands in the clip act as two enrolled "identities" (left/right half of frame)
    GAL.g = {}
    for name, sel in [('hand_left', lambda q, sh: (q[0] + q[2]) / 2 < sh[1] / 2), ('hand_right', lambda q, sh: (q[0] + q[2]) / 2 >= sh[1] / 2)]:
        cam2 = Camera(CFG['source']); codes = capture_palm_codes(cam2, CFG['enroll_frames'], max_frames=10 ** 4 if not SMOKE else 400, sel=sel)
        if len(codes) > CFG['gallery_templates']:
            r_ = GAL.enroll(name, codes, consent=True); REPORT.setdefault('enroll_selftest', {})[name] = r_; print('[enroll-selftest]', name, r_)
    # genuine/impostor decisions on frames → MCC of open-set accept decisions
    cam3 = Camera(CFG['source']); tp = fp = fn = tn = 0
    for _ in range(140):
        f = cam3.read()
        if f is None: break
        g = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY); b, s, c = student(g, TAU)
        for q in b[c == 1]:
            roi = palm_roi(g, q)
            if roi is None: continue
            truth = 'hand_left' if (q[0] + q[2]) / 2 < g.shape[1] / 2 else 'hand_right'; _, cand = GAL.identify(compcode(roi))
            for name, d, t in cand:
                acc, gen = d <= t, name == truth
                tp += acc and gen; fn += (not acc) and gen; fp += acc and not gen; tn += (not acc) and not gen
    REPORT['palm_id_selftest'] = dict(MCC=round(mcc(tp, fp, fn, tn), 4), TP=tp, FP=fp, FN=fn, TN=tn,
                                      note='two hands of one person in a demo clip — mechanics check, not a biometric benchmark')
    print('[palm-id] self-test', REPORT['palm_id_selftest'])
cam.close()

# ── §9 OpenAI-compatible /v1/chat/completions ──────────────────────────────────
from fastapi import FastAPI
from fastapi.responses import JSONResponse
import uvicorn
app = FastAPI(title='VG360-PALMID'); MODEL_ID = 'vg360-palmid-mono-int8'; _DEB = {}
@app.get('/v1/models')
def _models(): return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model', 'owned_by': 'innoura'}]}
@app.post('/v1/chat/completions')
def _chat(body: dict):
    results = []
    for m in body.get('messages', []):
        if m.get('role') == 'user' and isinstance(m.get('content'), list):
            for p in m['content']:
                if p.get('type') == 'image_url':
                    url = p['image_url']['url']; raw = base64.b64decode(url.split(',', 1)[1]) if url.startswith('data:') else urllib.request.urlopen(url, timeout=10).read()
                    g = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_GRAYSCALE); results.append(draw_and_identify(g, _DEB)[1])
    content = {'results': results} if results else {'enrolled': list(GAL.g), 'thresholds': dict(zip(CFG['classes'], TAU))}
    return JSONResponse(sanitize({'id': f'chatcmpl-{int(time.time()*1e3)}', 'object': 'chat.completion', 'created': int(time.time()), 'model': MODEL_ID,
        'choices': [{'index': 0, 'finish_reason': 'stop', 'message': {'role': 'assistant', 'content': json.dumps(sanitize(content))}}],
        'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}}))
def _serve():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
    loop.run_until_complete(uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=CFG['port'], log_level='warning', loop='asyncio')).serve())
threading.Thread(target=_serve, daemon=True).start()
for _ in range(50):
    try: urllib.request.urlopen(f"http://127.0.0.1:{CFG['port']}/v1/models", timeout=1); break
    except Exception: time.sleep(0.2)
g0 = SPLITS['val'][0][0] if SPLITS['val'] else SPLITS['train'][0][0]
req = {'model': MODEL_ID, 'messages': [{'role': 'user', 'content': [{'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,' + base64.b64encode(cv2.imencode('.png', g0)[1]).decode()}}]}]}
resp = json.loads(urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{CFG['port']}/v1/chat/completions", data=json.dumps(req).encode(),
                  headers={'Content-Type': 'application/json'}), timeout=30).read())
REPORT['endpoint'] = dict(url=f"http://0.0.0.0:{CFG['port']}/v1/chat/completions", selftest_objects=len(json.loads(resp['choices'][0]['message']['content'])['results'][0]))
json.dump(sanitize(REPORT), open(os.path.join(WORK, 'palmid_report.json'), 'w'), indent=1)
print('\n══ VG360-PALMID SUMMARY ══\n' + json.dumps(sanitize(REPORT), indent=1)[:3500])
print('Gallery:', GALLERY_PATH, '— delete someone with GAL.delete("name")')
